# 02 · Adding Metadata to Zarr
Reads Harmony acquisition metadata and assay layout XML for the Live-1 acquisition,
constructs OME-NGFF compliant `omero` and `multiscales` metadata blocks,
and writes them as `.zattrs` to both a top-level `OME.zarr` group and the zarr root.

Metadata includes: framerate, channel wavelengths, objective specs, pixel resolution,
Z spacing, and per-well experimental conditions from the assay layout.

In [ ]:
import os
import glob
import zarr
import pandas as pd
import numpy as np
from scipy import stats
from macrohet import dataio

## 1 · Load Harmony metadata and assay layout

In [ ]:
base_dir = '/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-1__2025-04-09T18_25_04-Measurement 1'

metadata = dataio.read_harmony_metadata(
    os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
)

assay_layout = dataio.read_harmony_metadata(
    os.path.join(base_dir, 'acquisition/Assaylayout/BPP0050_Live-cell to 4i-1.xml'),
    assay_layout=True
)

print(f"Metadata columns:         {list(metadata.keys())}")
print(f"Unique positions:         {len(metadata[['Row', 'Col']].drop_duplicates())}")
print(f"Assay layout rows:        {len(assay_layout)}")

## 2 · Inspect zarr array structure

In [ ]:
zarr_root_dir = os.path.join(base_dir, 'acquisition/zarr')
zarr_dirs     = glob.glob(os.path.join(zarr_root_dir, '*'))

zarr_group = zarr.open_group(zarr_dirs[0])
print(f"Example position: {zarr_dirs[0]}")
print(f"Image shape:      {zarr_group.images.shape}  # (T, C, Z, Y, X)")

## 3 · Build OME metadata
Constructs `omero` (framerate, channels, objective) and `multiscales` (axes, pixel spacing)
metadata blocks from the Harmony XML.

In [ ]:
omero_metadata = {}
average_time_difference_seconds = None

# --- Framerate ---
if 'AbsTime' in metadata.columns and 'TimepointID' in metadata.columns:
    try:
        if not pd.api.types.is_datetime64_any_dtype(metadata['AbsTime']):
            metadata['AbsTime'] = pd.to_datetime(metadata['AbsTime'], format='ISO8601', utc=True)

        timepoint_abs_times = (
            metadata[['TimepointID', 'AbsTime']]
            .drop_duplicates(subset=['TimepointID'])
            .sort_values(by='TimepointID', key=lambda x: pd.to_numeric(x, errors='ignore'))
        )
        time_diffs = timepoint_abs_times['AbsTime'].diff().dt.total_seconds().dropna()

        if not time_diffs.empty:
            average_time_difference_seconds = time_diffs.mean()
            if average_time_difference_seconds > 0:
                framerate = 1 / average_time_difference_seconds
                omero_metadata['frameRate'] = framerate
                print(f"Framerate: {framerate:.6f} fps  ({average_time_difference_seconds/3600:.2f} hr interval)")
        else:
            print("Warning: Could not calculate time differences for framerate.")
    except Exception as e:
        print(f"Warning: Framerate calculation failed: {e}")

# --- Channel metadata ---
channels_data = []
for _, row in metadata.drop_duplicates(subset=['ChannelID']).iterrows():
    emission_wavelength_meters   = None
    excitation_wavelength_meters = None

    if pd.notna(row['MainEmissionWavelength']):
        try:
            emission_wavelength_meters = float(row['MainEmissionWavelength']) * 1e-9
        except ValueError:
            print(f"Warning: Could not parse emission wavelength for channel {row['ChannelID']}")

    if pd.notna(row['MainExcitationWavelength']):
        try:
            excitation_wavelength_meters = float(row['MainExcitationWavelength']) * 1e-9
        except ValueError:
            print(f"Warning: Could not parse excitation wavelength for channel {row['ChannelID']}")

    channels_data.append({
        "id":                   int(row['ChannelID']),
        "label":                row['ChannelName'],
        "emissionWaveMeters":   emission_wavelength_meters,
        "excitationWaveMeters": excitation_wavelength_meters,
    })
omero_metadata['channels'] = channels_data

# --- Objective metadata ---
objective_data = {}
if 'ObjectiveMagnification' in metadata.columns and pd.notna(metadata['ObjectiveMagnification'].iloc[0]):
    try:
        objective_data['magnification'] = float(metadata['ObjectiveMagnification'].iloc[0])
    except ValueError:
        print("Warning: Could not parse objective magnification.")
if 'ObjectiveNA' in metadata.columns and pd.notna(metadata['ObjectiveNA'].iloc[0]):
    try:
        objective_data['numericalAperture'] = float(metadata['ObjectiveNA'].iloc[0])
    except ValueError:
        print("Warning: Could not parse objective NA.")
if objective_data:
    omero_metadata['objective'] = objective_data

# --- Multiscales axes with physical spacing ---
multiscales_data = [{
    "name":     "images",
    "datasets": [],
    "axes": [
        {"name": "t", "type": "time",    "unit": "second", "spacing": average_time_difference_seconds},
        {"name": "c", "type": "channel"},
        {"name": "z", "type": "space",   "unit": "meter",  "spacing": None},
        {"name": "y", "type": "space",   "unit": "meter",  "pixelResolution": None},
        {"name": "x", "type": "space",   "unit": "meter",  "pixelResolution": None}
    ],
    "type":    "image",
    "version": "0.4"
}]

# Populate dataset paths from existing zarr positions
for filename in os.listdir(zarr_root_dir):
    if filename.endswith('.zarr'):
        multiscales_data[0]['datasets'].append({"path": f"{filename}/images"})

# Pixel resolution (X, Y)
if 'ImageResolutionX' in metadata.columns and 'ImageResolutionY' in metadata.columns:
    try:
        multiscales_data[0]['axes'][3]['pixelResolution'] = float(metadata['ImageResolutionY'].iloc[0])
        multiscales_data[0]['axes'][4]['pixelResolution'] = float(metadata['ImageResolutionX'].iloc[0])
    except ValueError:
        print("Warning: Could not parse pixel resolution.")

# Z spacing (modal inter-plane distance)
if 'PositionZ' in metadata.columns:
    try:
        z_positions_sorted = np.sort(metadata['PositionZ'].astype(float).unique())
        if len(z_positions_sorted) > 1:
            z_diffs      = np.diff(z_positions_sorted)
            mode_result  = stats.mode(z_diffs)
            if mode_result.count > 0:
                multiscales_data[0]['axes'][2]['spacing'] = mode_result.mode
                multiscales_data[0]['axes'][2]['unit']    = "meter"
        else:
            print("Warning: Only one unique Z position found, cannot calculate Z spacing.")
    except ValueError:
        print("Warning: Could not process PositionZ for Z spacing.")

## 4 · Write metadata to OME.zarr

In [ ]:
top_level_attrs = {
    "omero":       omero_metadata,
    "multiscales": multiscales_data,
    "version":     "0.4"
}

output_zarr_path = os.path.join(base_dir, 'acquisition/OME.zarr')
group = zarr.group(output_zarr_path)
group.attrs.put(top_level_attrs)

print(f"OME-Zarr metadata saved to: {output_zarr_path}/.zattrs")

## 5 · Add assay layout to zarr root

In [ ]:
group = zarr.open_group(zarr_root_dir, mode='a')
group.attrs['assay_layout'] = assay_layout.to_dict(orient='index')
print(f"Assay layout metadata added to: {zarr_root_dir}/.zattrs")

## 6 · Verify metadata written correctly

In [ ]:
ome_group    = zarr.open_group(output_zarr_path, mode='r')
written_meta = ome_group.attrs.asdict()

print("Keys written:", list(written_meta.keys()))
print("\nChannels:")
for ch in written_meta['omero']['channels']:
    print(f"  {ch['id']}: {ch['label']}  ex={ch['excitationWaveMeters']}  em={ch['emissionWaveMeters']}")